In [ ]:
import duckdb
from pathlib import Path
from datetime import date

In [ ]:
con = duckdb.connect("../../ProjectData.duckdb")
con.execute("CREATE SCHEMA IF NOT EXISTS gold")

silver_files = sorted(Path("../../data/silver").glob("cleaned_data_load_date=*.parquet"))
if not silver_files:
    raise FileNotFoundError("No Silver Parquet found.")

silver_path = silver_files[-1].as_posix()
con.execute(f"""
    CREATE OR REPLACE VIEW silver_data AS
    SELECT * FROM read_parquet('{silver_path}')
""")
print(f"Silver: {silver_files[-1].name}")
print("Rows:", con.execute("SELECT COUNT(*) FROM silver_data").fetchone()[0])

In [ ]:
con.execute("DROP TABLE IF EXISTS gold.review_evidence_features")
con.execute("""
    CREATE TABLE gold.review_evidence_features AS
    WITH base AS (
        SELECT
            _index,
            product_id,
            product_parent,
            label,
            product_category_id,
            review_body,

            -- Sentence count approximation (min 1 to avoid division by zero)
            GREATEST(
                REGEXP_COUNT(review_body, '[.!?]+'), 1
            ) AS sentence_count_approx,

            -- Word count
            ARRAY_LENGTH(STRING_SPLIT(TRIM(review_body), ' ')) AS word_count,

            -- MEASUREMENT DENSITY
            -- Counts patterns like: 14h, 3.5 kg, 60cm, 500ml, 72°, 1080p, 3ft
            -- Physical measurements signal hands-on, empirical experience.
            REGEXP_COUNT(
                review_body,
                '\d+[\.,]?\d*\s{0,2}(cm|mm|km|mg|kg|gb|mb|tb|ml|cl|dl|hz|khz|mhz|ghz|mp|fps|dpi|ppi|rpm|watt|wh|mah|lm|db|nm|ft|inch|oz|lb|yd|°c|°f|°|%|p\b|h\b|min\b|sec\b|g\b|l\b|m\b|k\b)'
            ) AS measurement_count,

            -- PRICE REFERENCE DENSITY
            -- Captures: €12, $9.99, £50, ¥1000, 12€, 9.99$
            -- Price comparisons are a strong signal of purchase deliberation.
            REGEXP_COUNT(
                review_body,
                '[$€£¥]\s?\d+[\.,]?\d*|\d+[\.,]?\d*\s?[$€£¥]'
            ) AS price_ref_count,

            -- COMPARATIVE / ORDINAL DENSITY
            -- Captures: superlatives, ordinals, and explicit comparisons.
            -- Reviews that compare to alternatives provide relative context.
            REGEXP_COUNT(
                review_body,
                '\b(best|worst|better|worse|compared to|versus|vs\.?|unlike|superior|inferior|first|second|third|top|bottom|highest|lowest|greatest|least|most|fewer|more than|less than)\b'
            ) AS ordinal_comparison_count

        FROM silver_data
        WHERE review_body IS NOT NULL
          AND TRIM(review_body) != ''
    ),
    densities AS (
        SELECT
            _index,
            product_id,
            product_parent,
            label,
            product_category_id,
            measurement_count,
            price_ref_count,
            ordinal_comparison_count,

            -- Normalise each count by sentence count to produce densities
            measurement_count    * 1.0 / sentence_count_approx  AS measurement_density,
            price_ref_count      * 1.0 / sentence_count_approx  AS price_reference_density,
            ordinal_comparison_count * 1.0 / sentence_count_approx AS ordinal_comparison_density,

            -- Composite quantitative evidence score:
            -- Weighted sum: measurements carry the most discriminative weight
            -- because they require direct product interaction to report.
            -- Price references are second (deliberate purchase context).
            -- Comparisons are lowest weight as they can appear in any review type.
            (
                3.0 * measurement_count
              + 2.0 * price_ref_count
              + 1.0 * ordinal_comparison_count
            ) * 1.0 / NULLIF(sentence_count_approx, 0) AS quantitative_evidence_score

        FROM base
    ),
    normed AS (
        SELECT
            *,
            -- Z-score of composite score within product category
            -- Normalises for category-level differences in how often reviewers
            -- cite measurements (e.g. electronics vs music albums)
            (quantitative_evidence_score
                - AVG(quantitative_evidence_score) OVER (PARTITION BY product_category_id))
                / NULLIF(STDDEV(quantitative_evidence_score) OVER (PARTITION BY product_category_id), 0)
                AS evidence_cat_zscore
        FROM densities
    )
    SELECT * FROM normed
""")

print("gold.review_evidence_features created.")
print("Rows:", con.execute("SELECT COUNT(*) FROM gold.review_evidence_features").fetchone()[0])

In [ ]:
con.execute("""
    SELECT
        COUNT(*)                                                         AS total_rows,
        SUM(measurement_count)                                           AS total_measurements,
        SUM(price_ref_count)                                             AS total_price_refs,
        SUM(ordinal_comparison_count)                                    AS total_comparisons,
        ROUND(AVG(measurement_density),          4)                      AS avg_meas_density,
        ROUND(AVG(price_reference_density),      4)                      AS avg_price_density,
        ROUND(AVG(ordinal_comparison_density),   4)                      AS avg_ord_density,
        ROUND(AVG(quantitative_evidence_score),  4)                      AS avg_evidence_score,
        ROUND(MAX(quantitative_evidence_score),  4)                      AS max_evidence_score,
        SUM(CASE WHEN quantitative_evidence_score > 0 THEN 1 ELSE 0 END) AS reviews_with_evidence
    FROM gold.review_evidence_features
""").df()

In [ ]:
con.execute("""
    SELECT
        label,
        COUNT(*)                                             AS reviews,
        ROUND(AVG(measurement_density),        4)           AS avg_meas_density,
        ROUND(AVG(price_reference_density),    4)           AS avg_price_density,
        ROUND(AVG(ordinal_comparison_density), 4)           AS avg_ord_density,
        ROUND(AVG(quantitative_evidence_score), 4)          AS avg_evidence_score,
        ROUND(AVG(evidence_cat_zscore),        3)           AS avg_evidence_zscore
    FROM gold.review_evidence_features
    WHERE label IS NOT NULL
    GROUP BY label
    ORDER BY label DESC
""").df()

In [ ]:
con.execute("""
    SELECT
        product_category_id,
        COUNT(*)                                              AS reviews,
        ROUND(AVG(quantitative_evidence_score), 4)           AS avg_evidence_raw,
        ROUND(AVG(evidence_cat_zscore),         3)           AS avg_evidence_zscore
    FROM gold.review_evidence_features
    GROUP BY product_category_id
    ORDER BY avg_evidence_raw DESC
""").df()

In [ ]:
gold_dir = Path("../../data/gold")
gold_dir.mkdir(parents=True, exist_ok=True)

output_file = gold_dir / f"evidence_features_load_date={date.today().isoformat()}.parquet"
con.execute(f"""
    COPY (SELECT * FROM gold.review_evidence_features)
    TO '{output_file.as_posix()}' (FORMAT PARQUET)
""")
print(f"Saved: {output_file.resolve()}")

In [ ]:
con.close()